# 02 — Xây dựng dataset session đã xác minh

Đầu vào là bốn file `*_all.csv` do notebook 01 tạo. Đầu ra chính là `Data/sessions_verified.parquet`; notebook 03 chỉ train từ đúng file này.

Notebook loại thiết bị nằm trong `device_labels_excluded.csv`, áp taxonomy verified, kiểm tra feature không leak và ghi `Data/dataset_manifest.json` để các bước sau truy vết đúng dataset.


In [ ]:
# Nạp định nghĩa từ 03 mà không chạy train.
from pathlib import Path

_cwd = Path.cwd().resolve()
_ROOT = next((p for p in (_cwd, *_cwd.parents)
              if (p / "Code" / "03_train_model.ipynb").is_file()), None)
assert _ROOT is not None, f"Không thấy Code/03_train_model.ipynb quanh {_cwd}"
_CODE = _ROOT / "Code"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_CODE / "03_train_model.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import pandas as pd
from IPython.display import display

import json
import re

OUT_PATH = ROOT / 'Data' / 'device_alias_map.csv'
assert FEAT_DIR.exists(), f"Không tìm thấy {FEAT_DIR.resolve()} — chạy 01_feature_extract.ipynb trước"


def clean_name(name):
    """Gộp khoảng trắng thừa (vài tên IDLE có double space, ví dụ 'Gosund ESP_1ACEE1  Socket')."""
    return re.sub(r"\s+", " ", str(name)).strip()


def norm_mac(raw):
    """Cột client_mac là bootp.hw.mac_addr đã pad đủ 16 byte, chỉ 6 byte đầu là MAC thật."""
    hexs = re.sub(r"[^0-9a-f]", "", str(raw).lower())[:12]
    if len(hexs) != 12 or hexs == "0" * 12:
        return None
    return ":".join(hexs[i:i + 2] for i in range(0, 12, 2))


print("Python  :", __import__("sys").executable)
print("Bảng feature:", FEAT_DIR)

## 1. Lấy MAC đại diện cho từng (scenario, device)

DHCP là nguồn duy nhất trong 4 nguồn có MAC ở tầng ứng dụng — DNS/mDNS/TLS chỉ có IP, mà IP thì đổi theo
lease nên không dùng làm khoá được. May là 100% thiết bị đều có ít nhất 1 record DHCP.

Dùng `client_mac` (`bootp.hw.mac_addr`) chứ không dùng `src_mac`: với gói OFFER/ACK do server gửi thì
`src_mac` là MAC của router, còn `client_mac` luôn là thiết bị đang xin IP.

In [ ]:
dhcp = pd.read_csv(FEAT_DIR / "dhcp_features_all.csv", low_memory=False)
dhcp["device"] = dhcp.device.map(clean_name)
dhcp["mac"] = dhcp.client_mac.map(norm_mac)
dhcp = dhcp[dhcp.mac.notna()]

# Cảnh báo nếu 1 thiết bị lại gắn với nhiều MAC (dấu hiệu extractor gán nhầm traffic)
mac_sets = dhcp.groupby(["scenario", "device"]).mac.apply(set)
multi_mac = mac_sets[mac_sets.map(len) > 1]
for (scenario, device), macs in multi_mac.items():
    print(f"CẢNH BÁO nhiều MAC — {scenario} / {device}: {sorted(macs)}")

# MAC xuất hiện nhiều nhất làm đại diện
rep = (
    dhcp.groupby(["scenario", "device"]).mac
    .agg(lambda s: s.value_counts().idxmax())
    .rename("mac")
    .reset_index()
)

print(f"{len(rep)} cặp (scenario, device) — {rep.mac.nunique()} MAC duy nhất")
rep.head()

## 2. Chọn tên canonical

Hai tên cùng MAC ⇒ cùng một thiết bị vật lý. Ưu tiên **tên IDLE** làm canonical vì mô tả chính xác hơn:
đúng chính tả brand (`Teckin` chứ không phải `Tekin`), và nhóm Gosund được đặt theo ESP-ID gắn với MAC
(`Gosund ESP_10098F Socket`) thay vì theo vị trí đặt trong phòng lab (`Gosund Plug - Red`) — vị trí là
thuộc tính của lần capture, không phải của thiết bị.

Chỉ sửa tay vài tên có ký tự phá filename (`/`, `:`) hoặc viết hoa toàn bộ. Không đổi tên ngoài danh sách
này để map vẫn truy ngược được về dataset gốc.

In [ ]:
CANONICAL_OVERRIDE = {
    "AMCREST WiFi Camera": "Amcrest WiFi Camera",
    "Borun/Sichuan-AI Camera": "Borun Sichuan-AI Camera",
    "DCS8000LHA1 D-Link Mini Camera": "D-Link Mini Camera DCS-8000LHA1",
    "HeimVision SmartLife Radio/Lamp": "HeimVision SmartLife Radio-Lamp",
    "Ring Base Station AC:1236": "Ring Base Station",
    "SIMCAM 1S (AMPAKTec)": "SimCam 1S",
}

idle_by_mac = rep[rep.scenario == "IDLE"].set_index("mac").device
power_by_mac = rep[rep.scenario == "POWER"].set_index("mac").device

# Tên POWER làm fallback cho thiết bị không xuất hiện trong IDLE, còn lại IDLE ghi đè
mac2canonical = {mac: name for mac, name in power_by_mac.items()}
mac2canonical.update({mac: name for mac, name in idle_by_mac.items()})
mac2canonical = {mac: CANONICAL_OVERRIDE.get(name, name) for mac, name in mac2canonical.items()}

alias = rep.copy()
alias["canonical_device"] = alias.mac.map(mac2canonical)
alias["match_method"] = alias.mac.map(
    lambda m: "dhcp_mac" if m in idle_by_mac.index and m in power_by_mac.index else "single_scenario"
)

# In toàn bộ cặp POWER -> IDLE đã nối được để mắt người soát lại
pairs = alias.pivot_table(index="canonical_device", columns="scenario", values="device", aggfunc="first")
print(f"{(alias.match_method == 'dhcp_mac').sum() // 2} thiết bị nối được qua MAC, "
      f"{(alias.match_method == 'single_scenario').sum()} chỉ có ở một kịch bản\n")
pairs

## 3. Hợp nhất category

Lấy category từ `capture_summary_all.csv` (đầy đủ hơn file DHCP), gộp `Cameras` → `Camera`, rồi kiểm tra
xem sau khi hợp nhất có thiết bị nào bị gán 2 category khác nhau không.

In [ ]:
CATEGORY_OVERRIDE = {"Cameras": "Camera"}

summary = pd.read_csv(FEAT_DIR / "capture_summary_all.csv")
summary["device"] = summary.device.map(clean_name)

categories = (
    summary.groupby(["scenario", "device"]).category
    .agg(lambda s: s.mode().iat[0])
    .rename("orig_category")
    .reset_index()
)

alias = alias.merge(categories, on=["scenario", "device"], how="left")
alias["canonical_category"] = alias.orig_category.map(lambda c: CATEGORY_OVERRIDE.get(clean_name(c), clean_name(c)))

conflicts = alias.groupby("canonical_device").canonical_category.nunique()
for device in conflicts[conflicts > 1].index:
    got = sorted(alias.loc[alias.canonical_device == device, "canonical_category"].unique())
    print(f"CẢNH BÁO category xung đột — {device}: {got}")

print("Số thiết bị theo category sau hợp nhất:")
print(alias.groupby("canonical_category").canonical_device.nunique().to_string())

## 4. Kiểm tra độ phủ

Map được xây từ file DHCP nên phải xác nhận nó phủ hết tên thiết bị xuất hiện trong **cả 4** file feature
— nếu DNS/TLS có tên nào không nằm trong map thì lúc merge ở bước sau sẽ sinh nhãn `NaN` âm thầm.

`dns_features_all.csv` ~100 MB nên đọc theo chunk, chỉ lấy 2 cột cần thiết.

In [ ]:
FEATURE_FILES = [
    "capture_summary_all.csv",
    "dhcp_features_all.csv",
    "dns_features_all.csv",
    "tls_features_all.csv",
]

known_keys = set(zip(alias.scenario, alias.device))
all_ok = True

for fname in FEATURE_FILES:
    seen = set()
    for chunk in pd.read_csv(FEAT_DIR / fname, usecols=["scenario", "device"],
                             chunksize=500_000, low_memory=False):
        seen |= set(zip(chunk.scenario, chunk.device.map(clean_name)))
    missing = seen - known_keys
    all_ok &= not missing
    status = "OK" if not missing else f"THIẾU {len(missing)}: {sorted(missing)}"
    print(f"{fname:28s} {len(seen):3d} cặp (scenario, device)  ->  {status}")

assert all_ok, "Có tên thiết bị không nằm trong map — kiểm tra lại trước khi dùng ở bước sau"

## 5. Lưu map

Khoá merge là cặp `(scenario, orig_device)` — đúng những cột đã có sẵn trong mọi file feature.
Bước sau chỉ cần:

```python
alias = pd.read_csv(ROOT / "Data" / "device_alias_map.csv")
df = df.merge(alias[["scenario", "orig_device", "canonical_device", "canonical_category", "mac"]],
              left_on=["scenario", "device"], right_on=["scenario", "orig_device"], how="left")
```

In [ ]:
alias_out = (
    alias.rename(columns={"device": "orig_device"})
    [["scenario", "orig_device", "orig_category", "canonical_device", "canonical_category",
      "mac", "match_method"]]
    .sort_values(["canonical_device", "scenario"])
    .reset_index(drop=True)
)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
alias_out.to_csv(OUT_PATH, index=False)

n_orig = len(alias_out)
n_canon = alias_out.canonical_device.nunique()
print(f"Đã ghi {OUT_PATH.resolve()}")
print(f"{n_orig} tên gốc -> {n_canon} thiết bị thật (giảm {n_orig - n_canon} lớp trùng)")
alias_out.head(20)

# Bước 2 — Feature table mức session

Gộp feature thô (1 row/event) thành **1 row / (thiết bị, collection window)** — đơn vị mà model sẽ thấy
lúc chạy thật. `session_id` sẵn có đã đúng mức đó: IDLE là `<device>_<date>_<hour>`, POWER là một file
pcap.

## Phạm vi feature

Đúng 4 nguồn, mỗi nguồn lấy phần **định danh** chứ không lấy counter:

| Nguồn | Lấy gì | Dạng |
|---|---|---|
| DHCP option 55 | `param_req_list` — thứ tự option thiết bị xin | categorical + cờ từng option |
| DHCP option 60 | `vendor_class_id` — chuỗi firmware/stack | text (char n-gram) |
| DNS | token của `qry_name` | text |
| mDNS | token của `qry_name` (chỉ query) | text |
| TLS | SNI token | text |
| TLS | `cipher_suites` + `alpn` + `version` | categorical kiểu JA3 |

**Không lấy**: mọi counter (`dns_n`, histogram qtype, đếm DHCP message type), `hostname` (option 12),
và mọi thứ dựa trên timestamp/kích thước gói — driver không cung cấp chuỗi timing.

## Hai nguyên tắc

1. **Không vượt quá telemetry của driver.** Không dùng `src_ip`/`dst_ip`/`dst_port`, không dùng timestamp
   từng gói. mDNS chỉ lấy query (driver bỏ QR=1).
2. **Missing là trạng thái hợp lệ, không impute.** Thiết bị không nói TLS là một đặc điểm nhận dạng thật.
   Numeric → `0` kèm cột `has_*`; text/categorical → `"<missing>"`.

Domain và SNI là feature **chính**, không phải leak: lúc chạy thật router vẫn đọc được DNS query và SNI,
nên thiết bị gọi `dss-na.amazon.com` là bằng chứng hợp lệ về Amazon. Lý do vẫn đo riêng bản không có
chúng là **DoH/DoT và ECH** — khi DNS bị mã hoá và SNI bị ẩn thì cả hai biến mất cùng lúc. Nhóm
`no_content` trong `sdc.feature_sets()` phục vụ đúng ablation đó.

## 6. Nạp map, nhãn và khung session

Khung session = hợp của `session_id` trên cả 3 file nguồn. Phải lấy hợp chứ không lấy từ một file: thiết
bị chỉ có DHCP+DNS sẽ biến mất nếu xuất phát từ phía TLS.

`date` dùng làm **group key khi chia train/test** (mục 7 kế hoạch). Lấy từ cột `date` của capture chứ
không suy từ `ts`: capture IDLE kéo qua nửa đêm (`2021_12_23_Idle.pcap` có event tới 2021-12-24 09:59),
suy từ `ts` sẽ xé một capture thành hai group.

In [ ]:
import numpy as np

LABELS_PATH = ROOT / 'Data' / 'device_labels_verified.csv'
EXCLUDED_PATH = ROOT / 'Data' / 'device_labels_excluded.csv'
SESSIONS_PATH = ROOT / 'Data' / 'sessions_verified.parquet'

alias_map = pd.read_csv(OUT_PATH)
ALIAS = dict(zip(zip(alias_map.scenario, alias_map.orig_device.map(clean_name)),
                 alias_map.canonical_device))
MAC_OF = dict(zip(alias_map.canonical_device, alias_map.mac))

labels = pd.read_csv(LABELS_PATH).set_index("canonical_device")[["make", "type", "model"]]
excluded = set(pd.read_csv(EXCLUDED_PATH).canonical_device)
assert set(labels.index) | excluded == set(alias_map.canonical_device), "Nhãn/exclusion và alias map lệch danh sách thiết bị"
assert not set(labels.index) & excluded, "Thiết bị vừa verified vừa excluded"


def load_source(fname, usecols):
    """Đọc 1 file feature thô, gắn canonical_device + date. clean_name là bắt buộc —
    'Gosund ESP_1ACEE1  Socket' có hai dấu cách trong file gốc, map lưu bản đã gộp."""
    df = pd.read_csv(FEAT_DIR / fname, usecols=usecols + BASE_COLS, low_memory=False)
    df["device"] = df.device.map(clean_name)
    df["canonical_device"] = [ALIAS[(s, d)] for s, d in zip(df.scenario, df.device)]
    df = df[~df.canonical_device.isin(excluded)].copy()
    # POWER không có cột date -> lấy ngày từ ts (mỗi pcap POWER gọn trong 1 ngày)
    date = pd.to_datetime(df["date"], errors="coerce")
    fallback = pd.to_datetime(df["ts"], unit="s", errors="coerce").dt.normalize()
    df["date"] = date.fillna(fallback).dt.date.astype(str)
    return df


BASE_COLS = ["scenario", "device", "capture_file", "session_id", "ts", "date"]

dhcp_raw = load_source("dhcp_features_all.csv",
                       ["src_mac", "client_mac", "msg_type", "hostname",
                        "vendor_class_id", "param_req_list"])
dns_raw = load_source("dns_features_all.csv",
                      ["is_mdns", "is_response", "qry_name", "qry_type", "rcode"])
tls_raw = load_source("tls_features_all.csv",
                      ["tls_version", "sni", "alpn", "cipher_suites", "n_cipher_suites"])
expected_mac = dhcp_raw.canonical_device.map(MAC_OF)
dhcp_raw = dhcp_raw[(dhcp_raw.src_mac.map(norm_mac) == expected_mac)
                    | (dhcp_raw.client_mac.map(norm_mac) == expected_mac)]
dhcp_raw = dhcp_raw[dhcp_raw[["vendor_class_id", "param_req_list"]].notna().any(axis=1)].copy()
dns_raw = dns_raw[dns_raw.is_response != True].copy()  # DNS và mDNS chỉ dùng query

# Khung session: hợp của cả 3 nguồn, 1 dòng / session_id
frame = (
    pd.concat([d[["session_id", "canonical_device", "scenario", "capture_file", "date"]]
               for d in (dhcp_raw, dns_raw, tls_raw)])
    .drop_duplicates("session_id")
    .set_index("session_id")
    .sort_index()
)
frame["mac"] = frame.canonical_device.map(MAC_OF)

print(f"{len(frame)} session — {frame.canonical_device.nunique()} thiết bị, "
      f"{frame.date.nunique()} ngày")
print(frame.scenario.value_counts().to_string())

## 7. DHCP — option 55 và option 60

`param_req_list` (option 55) giữ **hai dạng**: nguyên chuỗi làm categorical, và tách từng option thành
cờ nhị phân. Chuỗi nguyên bản bắt được cả *thứ tự* option — mỗi stack TCP/IP xin theo thứ tự riêng nên
thứ tự mang thông tin, **không được sort**. Cờ nhị phân bắt trường hợp hai danh sách chỉ khác nhau một
hai option, thứ mà categorical coi là hai giá trị hoàn toàn khác.

`vendor_class_id` (option 60) là chuỗi tự do kiểu `dhcpcd-6.8.2:Linux-4.4.22+:armv7l:MT8167B` — để
nguyên cho TF-IDF char n-gram ở bước train, vì phần có ích nằm ở chuỗi con (`dhcpcd`, `Linux`, `armv7l`)
chứ không phải giá trị đầy đủ.

In [ ]:
MISSING = "<missing>"


def mode_of(series):
    """Giá trị hay gặp nhất, bỏ NaN. Trả MISSING nếu cả session không có giá trị nào."""
    s = series.dropna()
    return s.mode().iat[0] if len(s) else MISSING


def parse_opts(value):
    if not isinstance(value, str):
        return []
    return [int(x) for x in re.findall(r"\d+", value)]


g = dhcp_raw.groupby("session_id")
dhcp_f = pd.DataFrame(index=g.size().index)
dhcp_f["dhcp_prl"] = g.param_req_list.agg(mode_of)     # option 55
dhcp_f["dhcp_vci"] = g.vendor_class_id.agg(mode_of)    # option 60

all_opts = sorted({o for v in dhcp_raw.param_req_list.dropna().unique() for o in parse_opts(v)})
opt_sets = dhcp_f.dhcp_prl.map(lambda v: set(parse_opts(v)))
for o in all_opts:
    dhcp_f[f"dhcp_opt_{o}"] = opt_sets.map(lambda s, o=o: int(o in s))
dhcp_f["dhcp_prl_len"] = opt_sets.map(len)

print(f"{len(dhcp_f)} session có DHCP")
print(f"  option 55: {dhcp_f.dhcp_prl.nunique()} giá trị, {len(all_opts)} option riêng biệt {all_opts}")
print(f"  option 60: {dhcp_f.dhcp_vci.nunique()} giá trị")
print(f"    ví dụ: {[v for v in dhcp_f.dhcp_vci.unique() if v != MISSING][:3]}")
dhcp_f.head(3)

## 8. DNS và mDNS — token tên miền

Tách tên miền theo `.` và `-` (**không** tách theo `_`, vì nhãn service của mDNS như `_googlecast`,
`_hap`, `_tcp` cần giữ nguyên dấu gạch dưới để phân biệt với từ thường).

Mỗi session giữ **token duy nhất**, không giữ tần suất: một thiết bị hỏi cùng một domain 500 lần sẽ làm
TF nở ra vô nghĩa. Giới hạn `MAX_TOKENS` để thiết bị nói nhiều không lấn át thiết bị nói ít.

Hai nguồn tách riêng vì driver đối xử khác nhau: DNS unicast/53 có payload query, còn mDNS **chỉ có
query** (driver bỏ QR=1). Đo trên dữ liệu này thì lọc response không mất session nào — toàn bộ bản ghi
mDNS trong CIC vốn đã là query — nhưng vẫn lọc tường minh để code khớp với driver.

In [ ]:
MAX_TOKENS = 40     # trần token/session, chặn thiết bị nói nhiều lấn át


def session_domain_tokens(series, max_tokens=MAX_TOKENS):
    """Tên miền -> chuỗi token duy nhất, giữ thứ tự xuất hiện.
    Tách theo '.' và '-' nhưng KHÔNG theo '_': '_googlecast'/'_tcp' phải còn nguyên."""
    tokens = []
    for name in series.dropna():
        tokens.extend(t for t in re.split(r"[.\-]", str(name).lower().rstrip(".")) if t)
    unique = list(dict.fromkeys(tokens))[:max_tokens]
    return " ".join(unique) if unique else MISSING


# ---- DNS unicast/53 ----
dns = dns_raw[dns_raw.is_mdns != True]
dns_f = pd.DataFrame({"dns_tokens": dns.groupby("session_id").qry_name.agg(session_domain_tokens)})

# ---- mDNS: chỉ query, đúng như driver ----
mdns = dns_raw[(dns_raw.is_mdns == True) & (dns_raw.is_response != True)]
mdns_f = pd.DataFrame({"mdns_tokens": mdns.groupby("session_id").qry_name.agg(session_domain_tokens)})

n_mdns_all = int((dns_raw.is_mdns == True).sum())
print(f"DNS  : {len(dns_f):5d} session")
print(f"mDNS : {len(mdns_f):5d} session  (lọc response bỏ {n_mdns_all - len(mdns)} bản ghi)")
print(f"\nVí dụ token DNS : {dns_f.dns_tokens.iloc[0][:90]}")
print(f"Ví dụ token mDNS: {mdns_f.mdns_tokens.iloc[0][:90]}")

## 9. TLS — SNI và fingerprint handshake

Hai phần tách bạch, vì chúng hỏng theo hai cách khác nhau:

- **SNI** → token, giống DNS. Biến mất khi bật ECH.
- **`version | cipher_suites | alpn`** → fingerprint kiểu JA3, categorical. Vẫn còn kể cả khi ECH bật,
  nên đây là phần "chịu được mã hoá".

`tls_version` phải là **categorical**: `0303` là mã của TLS 1.2, pandas đọc thành số 303. Để numeric thì
model được phép so sánh `version > 200` — vô nghĩa. Nên ép lại thành chuỗi 4 chữ số.

Extractor dùng TCP reassembly; `tls_sni_tokens` chỉ bằng `<missing>` khi ClientHello thực sự không có SNI.
TLS GREASE được loại trước khi tạo fingerprint để cùng một client không sinh khoá ngẫu nhiên.

In [ ]:
def version_code(value):
    """'0303' trong file gốc bị pandas đọc thành số 303 — trả lại dạng mã 4 chữ số."""
    try:
        return f"{int(value):04d}"
    except (TypeError, ValueError):
        return MISSING


tls_raw["tls_version_code"] = tls_raw.tls_version.map(version_code)
tls_raw["tls_fp"] = (tls_raw.tls_version_code + "|"
                     + tls_raw.cipher_suites.astype(str) + "|"
                     + tls_raw.alpn.fillna("").astype(str))

g = tls_raw.groupby("session_id")
tls_f = pd.DataFrame(index=g.size().index)
tls_f["tls_fp"] = g.tls_fp.agg(mode_of)
tls_f["tls_version"] = g.tls_version_code.agg(mode_of)
tls_f["tls_alpn"] = g.alpn.agg(mode_of)
tls_f["tls_ciphers"] = g.cipher_suites.agg(mode_of)
tls_f["tls_sni_tokens"] = g.sni.agg(session_domain_tokens)

no_sni = (tls_f.tls_sni_tokens == MISSING).sum()
print(f"TLS: {len(tls_f)} session, {tls_f.tls_fp.nunique()} fingerprint duy nhất")
print(f"  version: {sorted(tls_f.tls_version.unique())}")
print(f"  có ClientHello nhưng không đọc được SNI: {no_sni}/{len(tls_f)} "
      f"({no_sni/len(tls_f)*100:.0f}%) — thiếu TCP reassembly")
print(f"\nVí dụ token SNI: {tls_f.tls_sni_tokens.iloc[0][:90]}")

## 10. Ghép, mask và gắn nhãn

Join `how="left"` **xuất phát từ `frame`** (hợp của mọi session), không xuất phát từ phía TLS/DNS — nếu
không, 4 thiết bị chỉ có DHCP+DNS sẽ biến mất khỏi tập train.

Cột `has_*` phải tính **trước** khi fillna, vì sau đó không còn phân biệt được "không có nguồn" với
"có nguồn nhưng giá trị bằng 0".

In [ ]:
# Convert raw event tables back to the canonical record contract, then call the
# exact same aggregate() implementation used by production inference.
records_by_session = {session_id: [] for session_id in frame.index}
for event in dhcp_raw.itertuples():
    records_by_session[event.session_id].append({
        "proto": "dhcp", "opt55": event.param_req_list,
        "opt60": event.vendor_class_id,
    })
for event in dns_raw.itertuples():
    if event.is_response == True:
        continue
    records_by_session[event.session_id].append({
        "proto": "mdns" if event.is_mdns == True else "dns",
        "qname": event.qry_name,
    })
for event in tls_raw.itertuples():
    records_by_session[event.session_id].append({
        "proto": "tls", "version": event.tls_version_code,
        "ciphers": event.cipher_suites, "alpn": event.alpn, "sni": event.sni,
    })

aggregate_cols = (
    ["has_dhcp", "dhcp_prl", "dhcp_vci"]
    + [f"dhcp_opt_{option}" for option in all_opts]
    + ["dhcp_prl_len", "has_dns", "dns_tokens", "has_mdns",
       "mdns_tokens", "has_tls", "tls_fp", "tls_version",
       "tls_alpn", "tls_ciphers", "tls_sni_tokens"]
)
feature_rows = [aggregate(records_by_session[session_id], aggregate_cols)
                for session_id in frame.index]
feature_frame = pd.DataFrame(feature_rows, index=frame.index)
sessions = frame.join(feature_frame, how="left")

META = ["canonical_device", "scenario", "capture_file", "date", "mac"]
sessions = sessions.join(labels, on="canonical_device")

# Cột hằng số không mang thông tin nhưng vẫn chiếm chỗ trong input tensor và model contract
LABELS = ["make", "type", "model"]
const = [c for c in sessions.columns
         if c not in META + LABELS + ["n_sources"] and sessions[c].nunique(dropna=False) <= 1]
if const:
    print(f"Bỏ {len(const)} cột hằng số: {const}")
    sessions = sessions.drop(columns=const)

assert sessions[LABELS].notna().all().all(), "Có session thiếu nhãn"
assert len(sessions) == len(frame), "Join làm đổi số dòng — có session_id trùng trong block feature"

print(f"\n{len(sessions)} session × {sessions.shape[1]} cột")
print("\nĐộ phủ nguồn (theo session):")
for c in ["has_dhcp", "has_dns", "has_mdns", "has_tls"]:
    print(f"  {c:10s} {sessions[c].mean()*100:5.1f}%")
print(f"\nSố nguồn/session: {sessions.n_sources.value_counts().sort_index().to_dict()}")
print(f"Số lớp — make {sessions['make'].nunique()}, "
      f"type {sessions['type'].nunique()}, model {sessions.model.nunique()}")

## 11. Nhóm feature và kiểm tra khớp telemetry

Nhóm feature giờ được suy từ `sdc.feature_sets()` — **một định nghĩa duy nhất** dùng chung cho mọi
notebook, thay vì mỗi notebook tự suy lại rồi lệch nhau.

| Nhóm | Dùng để làm gì |
|---|---|
| `text` | Cột token → TF-IDF ở bước train (vocab fit trên fold train) |
| `cat` | Categorical → vocab đóng băng + bucket OOV |
| `num` | Cờ option + mask, dùng thẳng |
| `source` | Ablation theo nguồn |
| `no_content` | Ablation **DoH/ECH**: bỏ `dns_tokens` + `tls_sni_tokens` |

`assert_serveable()` chặn lỗi nguy hiểm nhất của dự án: một feature lọt vào model mà driver không cung
cấp được lúc chạy thật.

In [ ]:

FS = feature_sets(sessions)
assert_serveable(FS["all"])

print(f"{len(FS['all'])} feature")
for group in ("num", "cat", "text"):
    print(f"  {group:5s} {len(FS[group]):3d}  {FS[group] if group != 'num' else ''}")
print(f"\nTheo nguồn:")
for src, cols in FS["source"].items():
    print(f"  {src:5s} {len(cols):3d}  {cols if len(cols) <= 6 else cols[:3] + ['...']}")
print(f"\nno_content (kịch bản DoH/ECH): {len(FS['no_content'])} feature "
      f"— bỏ {ENCRYPTED_RISK}")

print("\nĐộ phủ từng cột text (tỉ lệ session có giá trị thật):")
for c in FS["text"]:
    print(f"  {c:16s} {(sessions[c] != MISSING).mean()*100:5.1f}%")

## 12. Kiểm tra chống leak và lưu

Ba kiểm tra trước khi lưu, tương ứng ba cách hỏng đã biết:

1. **Không có cột nào tương quan 1-1 với nhãn theo kiểu định danh** (ví dụ lỡ để `mac` hay `session_id`
   lọt vào feature) — model sẽ đạt 100% và vô dụng.
2. **Group key phải tách được**: mỗi `date` chỉ thuộc một fold; kiểm tra không có ngày nào chứa cả POWER
   lẫn IDLE.
3. **Phân bố lớp**: lớp quá ít mẫu sẽ không học được, cần biết trước để đọc macro-F1 cho đúng.

In [ ]:
# 1. Không feature nào định danh thiết bị một cách tầm thường
assert "mac" not in FS["all"]
for col in FS["cat"]:
    per_val = sessions.groupby(col).canonical_device.nunique()
    if len(per_val) > 20 and (per_val == 1).all():
        print(f"NGHI VẤN — {col}: {len(per_val)} giá trị, mỗi giá trị chỉ ứng 1 thiết bị")

# 2. Group key tách được
mixed = sessions.groupby("date").scenario.nunique()
assert (mixed == 1).all(), f"Ngày có cả POWER lẫn IDLE: {mixed[mixed > 1].index.tolist()}"
print(f"Group key `date`: {sessions.date.nunique()} ngày "
      f"({sessions[sessions.scenario == 'POWER'].date.nunique()} POWER / "
      f"{sessions[sessions.scenario == 'IDLE'].date.nunique()} IDLE)")

# 3. Lớp hiếm
print(f"\nLớp có < {RARE_THRESHOLD} session (không học được, tách riêng khi report macro-F1):")
for head in LABEL_COLS:
    vc = sessions[head].value_counts()
    rare = vc[vc < RARE_THRESHOLD]
    print(f"  {head:7s} {len(vc):3d} lớp"
          + (f", hiếm: {rare.to_dict()}" if len(rare) else ", không có lớp hiếm"))

print("\nSession/thiết bị:")
per_dev = sessions.canonical_device.value_counts()
print(f"  min {per_dev.min()}  median {int(per_dev.median())}  max {per_dev.max()}")
print(f"  thiết bị < 10 session: {per_dev[per_dev < 10].to_dict()}")

sessions.to_parquet(SESSIONS_PATH)
print(f"\nĐã ghi {SESSIONS_PATH.resolve()}  ({len(sessions)} × {sessions.shape[1]})")

## 13. Ghi dataset manifest cho notebook 03

Manifest chứa checksum của dataset và taxonomy. Nhờ đó một candidate luôn truy được chính xác dữ liệu đã dùng để train.


In [ ]:
import hashlib
import json
from datetime import datetime

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

dataset_manifest = {
    'format': 'sdc-dataset-v1',
    'created': datetime.now().isoformat(timespec='seconds'),
    'sessions': str(SESSIONS_PATH.relative_to(ROOT)).replace('\\', '/'),
    'sessions_sha256': sha256(SESSIONS_PATH),
    'taxonomy': str(LABELS_PATH.relative_to(ROOT)).replace('\\', '/'),
    'taxonomy_sha256': sha256(LABELS_PATH),
    'excluded_sha256': sha256(EXCLUDED_PATH),
    'n_sessions': int(len(sessions)),
    'n_devices': int(sessions.canonical_device.nunique()),
    'heads': {h: int(sessions[h].nunique()) for h in LABEL_COLS},
}
manifest_path = ROOT / 'Data' / 'dataset_manifest.json'
manifest_path.write_text(json.dumps(dataset_manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print('Đã ghi:', manifest_path)
display(pd.DataFrame([dataset_manifest]))